In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set up display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")

# Define paths
raw_data_path = Path('../data/raw')
cleaned_data_path = Path('../data/cleaned')
cleaned_data_path.mkdir(exist_ok=True)

print("=" * 80)
print("DATA CLEANING & INTEGRATION: Learning Behavior Analytics Project")
print("=" * 80)

DATA CLEANING & INTEGRATION: Learning Behavior Analytics Project


In [2]:
# 1. LOAD ALL RAW DATA
print("\n1. LOADING RAW DATA")
print("-" * 80)

df_courses = pd.read_csv(raw_data_path / 'courses.csv')
df_student_info = pd.read_csv(raw_data_path / 'studentInfo.csv')
df_student_registration = pd.read_csv(raw_data_path / 'studentRegistration.csv')
df_student_vle = pd.read_csv(raw_data_path / 'studentVle.csv')
df_assessments = pd.read_csv(raw_data_path / 'assessments.csv')
df_student_assessment = pd.read_csv(raw_data_path / 'studentAssessment.csv')
df_vle = pd.read_csv(raw_data_path / 'vle.csv')

print(f"✓ All raw datasets loaded successfully!")
print(f"\nDatasets info:")
print(f"  - courses: {df_courses.shape[0]} rows × {df_courses.shape[1]} columns")
print(f"  - studentInfo: {df_student_info.shape[0]} rows × {df_student_info.shape[1]} columns")
print(f"  - studentRegistration: {df_student_registration.shape[0]} rows × {df_student_registration.shape[1]} columns")
print(f"  - studentVle: {df_student_vle.shape[0]} rows × {df_student_vle.shape[1]} columns")
print(f"  - assessments: {df_assessments.shape[0]} rows × {df_assessments.shape[1]} columns")
print(f"  - studentAssessment: {df_student_assessment.shape[0]} rows × {df_student_assessment.shape[1]} columns")
print(f"  - vle: {df_vle.shape[0]} rows × {df_vle.shape[1]} columns")


1. LOADING RAW DATA
--------------------------------------------------------------------------------
✓ All raw datasets loaded successfully!

Datasets info:
  - courses: 22 rows × 3 columns
  - studentInfo: 32593 rows × 12 columns
  - studentRegistration: 32593 rows × 5 columns
  - studentVle: 10655280 rows × 6 columns
  - assessments: 206 rows × 6 columns
  - studentAssessment: 173912 rows × 5 columns
  - vle: 6364 rows × 6 columns


In [3]:
# 2. DATA QUALITY ASSESSMENT
print("\n\n2. DATA QUALITY ASSESSMENT (BEFORE CLEANING)")
print("-" * 80)

def assess_data_quality(df, name):
    """Assess data quality of a dataset"""
    print(f"\n{name}:")
    print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"  Duplicates: {df.duplicated().sum()}")
    print(f"  Missing values: {df.isnull().sum().sum()}")
    null_pct = (df.isnull().sum() / len(df) * 100)
    if (null_pct > 0).any():
        print(f"  Column-wise missing %:")
        print(null_pct[null_pct > 0])

assess_data_quality(df_student_info, "Student Info")
assess_data_quality(df_student_registration, "Student Registration")
assess_data_quality(df_student_vle, "Student VLE")
assess_data_quality(df_assessments, "Assessments")
assess_data_quality(df_student_assessment, "Student Assessment")
assess_data_quality(df_vle, "VLE")
assess_data_quality(df_courses, "Courses")



2. DATA QUALITY ASSESSMENT (BEFORE CLEANING)
--------------------------------------------------------------------------------

Student Info:
  Shape: 32593 rows × 12 columns
  Duplicates: 0
  Missing values: 1111
  Column-wise missing %:
imd_band    3.408707
dtype: float64

Student Registration:
  Shape: 32593 rows × 5 columns
  Duplicates: 0
  Missing values: 22566
  Column-wise missing %:
date_registration       0.138066
date_unregistration    69.097659
dtype: float64

Student VLE:
  Shape: 10655280 rows × 6 columns
  Duplicates: 787170
  Missing values: 0

Assessments:
  Shape: 206 rows × 6 columns
  Duplicates: 0
  Missing values: 11
  Column-wise missing %:
date    5.339806
dtype: float64

Student Assessment:
  Shape: 173912 rows × 5 columns
  Duplicates: 0
  Missing values: 173
  Column-wise missing %:
score    0.099476
dtype: float64

VLE:
  Shape: 6364 rows × 6 columns
  Duplicates: 0
  Missing values: 10486
  Column-wise missing %:
week_from    82.385292
week_to      82.3852

In [4]:
# 3. CLEAN STUDENT INFO TABLE
print("\n\n3. CLEANING STUDENT INFO TABLE")
print("-" * 80)

df_student_info_clean = df_student_info.copy()

# Remove duplicates if any
initial_rows = len(df_student_info_clean)
df_student_info_clean = df_student_info_clean.drop_duplicates(subset=['id_student'], keep='first')
print(f"✓ Removed {initial_rows - len(df_student_info_clean)} duplicate student records")

# Handle missing values in student info
print(f"\nMissing values before cleaning:")
print(df_student_info_clean.isnull().sum())

# For final_result - check for NaN
if df_student_info_clean['final_result'].isnull().any():
    print(f"\nFound {df_student_info_clean['final_result'].isnull().sum()} missing final_result values")
    # Keep all records - missing values will be handled in feature engineering

# Standardize data types
df_student_info_clean['gender'] = df_student_info_clean['gender'].astype('category')
df_student_info_clean['region'] = df_student_info_clean['region'].astype('category')
df_student_info_clean['highest_education'] = df_student_info_clean['highest_education'].astype('category')
df_student_info_clean['age_band'] = df_student_info_clean['age_band'].astype('category')
df_student_info_clean['final_result'] = df_student_info_clean['final_result'].astype('category')

print(f"\n✓ Student Info cleaned: {len(df_student_info_clean)} records")
print(f"Data types standardized to appropriate categories")
print(f"\nSample of cleaned data:")
print(df_student_info_clean.head())



3. CLEANING STUDENT INFO TABLE
--------------------------------------------------------------------------------
✓ Removed 3808 duplicate student records

Missing values before cleaning:
code_module               0
code_presentation         0
id_student                0
gender                    0
region                    0
highest_education         0
imd_band                971
age_band                  0
num_of_prev_attempts      0
studied_credits           0
disability                0
final_result              0
dtype: int64

✓ Student Info cleaned: 28785 records
Data types standardized to appropriate categories

Sample of cleaned data:
  code_module code_presentation  id_student gender                region  \
0         AAA             2013J       11391      M   East Anglian Region   
1         AAA             2013J       28400      F              Scotland   
2         AAA             2013J       30268      F  North Western Region   
3         AAA             2013J       31604  

In [5]:
# 4. CLEAN STUDENT REGISTRATION TABLE
print("\n\n4. CLEANING STUDENT REGISTRATION TABLE")
print("-" * 80)

df_student_registration_clean = df_student_registration.copy()

# Remove duplicates
initial_rows = len(df_student_registration_clean)
df_student_registration_clean = df_student_registration_clean.drop_duplicates(
    subset=['id_student', 'code_module', 'code_presentation'], keep='first')
print(f"✓ Removed {initial_rows - len(df_student_registration_clean)} duplicate registrations")

# Handle missing date_registration values
print(f"\nMissing values:")
print(df_student_registration_clean[['date_registration', 'date_unregistration']].isnull().sum())

# Fill missing date_registration with median (by module-presentation)
for module in df_student_registration_clean['code_module'].unique():
    for presentation in df_student_registration_clean['code_presentation'].unique():
        mask = (df_student_registration_clean['code_module'] == module) & \
               (df_student_registration_clean['code_presentation'] == presentation)
        median_date = df_student_registration_clean.loc[mask, 'date_registration'].median()
        df_student_registration_clean.loc[mask & df_student_registration_clean['date_registration'].isnull(), 'date_registration'] = median_date

# date_unregistration filled with NaN is OK (represents active students)
print(f"\n✓ Student Registration cleaned: {len(df_student_registration_clean)} records")
print(f"Missing date_registration values filled with module-presentation median")

# Validate date logic
invalid_dates = (df_student_registration_clean['date_unregistration'] < df_student_registration_clean['date_registration']).sum()
print(f"Invalid date relationships (unregistration < registration): {invalid_dates}")

if invalid_dates > 0:
    print("Fixing invalid date relationships...")
    df_student_registration_clean.loc[
        df_student_registration_clean['date_unregistration'] < df_student_registration_clean['date_registration'],
        'date_unregistration'
    ] = np.nan



4. CLEANING STUDENT REGISTRATION TABLE
--------------------------------------------------------------------------------
✓ Removed 0 duplicate registrations

Missing values:
date_registration         45
date_unregistration    22521
dtype: int64

✓ Student Registration cleaned: 32593 records
Missing date_registration values filled with module-presentation median
Invalid date relationships (unregistration < registration): 21
Fixing invalid date relationships...


In [6]:
# 5. CLEAN VLE TABLE
print("\n\n5. CLEANING VLE TABLE")
print("-" * 80)

df_vle_clean = df_vle.copy()

# Remove duplicates
initial_rows = len(df_vle_clean)
df_vle_clean = df_vle_clean.drop_duplicates(
    subset=['id_site', 'code_module', 'code_presentation'], keep='first')
print(f"✓ Removed {initial_rows - len(df_vle_clean)} duplicate VLE site records")

print(f"\nMissing values:")
print(df_vle_clean[['week_from', 'week_to']].isnull().sum())

# Fill missing week values with 0 (unknown week)
df_vle_clean['week_from'] = df_vle_clean['week_from'].fillna(0).astype(int)
df_vle_clean['week_to'] = df_vle_clean['week_to'].fillna(0).astype(int)

# Validate week relationships: week_from should be <= week_to
invalid_weeks = (df_vle_clean['week_from'] > df_vle_clean['week_to']).sum()
print(f"Invalid week relationships (week_from > week_to): {invalid_weeks}")

if invalid_weeks > 0:
    print("Fixing invalid week relationships...")
    # Swap week_from and week_to where they're reversed
    mask = df_vle_clean['week_from'] > df_vle_clean['week_to']
    df_vle_clean.loc[mask, ['week_from', 'week_to']] = df_vle_clean.loc[mask, ['week_to', 'week_from']].values

# Standardize activity type
df_vle_clean['activity_type'] = df_vle_clean['activity_type'].astype('category')

print(f"\n✓ VLE cleaned: {len(df_vle_clean)} records")
print(f"Activity types: {df_vle_clean['activity_type'].nunique()}")
print(f"Week range: {df_vle_clean['week_from'].min()} - {df_vle_clean['week_to'].max()}")



5. CLEANING VLE TABLE
--------------------------------------------------------------------------------
✓ Removed 0 duplicate VLE site records

Missing values:
week_from    5243
week_to      5243
dtype: int64
Invalid week relationships (week_from > week_to): 0

✓ VLE cleaned: 6364 records
Activity types: 20
Week range: 0 - 29


In [7]:
# 6. CLEAN STUDENT VLE INTERACTIONS TABLE
print("\n\n6. CLEANING STUDENT VLE INTERACTIONS TABLE")
print("-" * 80)

df_student_vle_clean = df_student_vle.copy()

print(f"Initial shape: {df_student_vle_clean.shape}")

# Validate data
print(f"\nData validation:")
print(f"  Sum clicks - Min: {df_student_vle_clean['sum_click'].min()}, Max: {df_student_vle_clean['sum_click'].max()}")
print(f"  Date range: {df_student_vle_clean['date'].min()} to {df_student_vle_clean['date'].max()}")

# Remove records with 0 clicks (noise)
initial_rows = len(df_student_vle_clean)
df_student_vle_clean = df_student_vle_clean[df_student_vle_clean['sum_click'] > 0]
print(f"\n✓ Removed {initial_rows - len(df_student_vle_clean)} records with 0 clicks")

# Handle outliers in sum_click - using IQR method
Q1 = df_student_vle_clean['sum_click'].quantile(0.25)
Q3 = df_student_vle_clean['sum_click'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_student_vle_clean[
    (df_student_vle_clean['sum_click'] < lower_bound) | 
    (df_student_vle_clean['sum_click'] > upper_bound)
]
print(f"\nOutlier detection (IQR method):")
print(f"  Lower bound: {lower_bound:.0f}, Upper bound: {upper_bound:.0f}")
print(f"  Outliers detected: {len(outliers)} records")
print(f"  Outlier range: {outliers['sum_click'].min()} - {outliers['sum_click'].max()}")

# Keep outliers (they may be valid - important activities)
# But cap extreme values at 99th percentile for analysis
p99 = df_student_vle_clean['sum_click'].quantile(0.99)
print(f"  99th percentile: {p99:.0f}")

print(f"\n✓ Student VLE cleaned: {len(df_student_vle_clean)} records")
print(f"Missing values: {df_student_vle_clean.isnull().sum().sum()}")



6. CLEANING STUDENT VLE INTERACTIONS TABLE
--------------------------------------------------------------------------------
Initial shape: (10655280, 6)

Data validation:
  Sum clicks - Min: 1, Max: 6977
  Date range: -25 to 269

✓ Removed 0 records with 0 clicks

Outlier detection (IQR method):
  Lower bound: -2, Upper bound: 6
  Outliers detected: 1314646 records
  Outlier range: 7 - 6977
  99th percentile: 34

✓ Student VLE cleaned: 10655280 records
Missing values: 0


In [8]:
# 7. CLEAN ASSESSMENTS TABLE
print("\n\n7. CLEANING ASSESSMENTS TABLE")
print("-" * 80)

df_assessments_clean = df_assessments.copy()

# Remove duplicates
initial_rows = len(df_assessments_clean)
df_assessments_clean = df_assessments_clean.drop_duplicates(
    subset=['id_assessment', 'code_module', 'code_presentation'], keep='first')
print(f"✓ Removed {initial_rows - len(df_assessments_clean)} duplicate assessment records")

print(f"\nMissing values:")
print(df_assessments_clean.isnull().sum())

# Fill missing dates with 0 (unknown date)
df_assessments_clean['date'] = df_assessments_clean['date'].fillna(0).astype(int)

# Validate weight values
print(f"\nWeight validation:")
print(f"  Min: {df_assessments_clean['weight'].min()}, Max: {df_assessments_clean['weight'].max()}")
print(f"  Sum by module:")
print(df_assessments_clean.groupby('code_module')['weight'].sum())

# Standardize assessment type
df_assessments_clean['assessment_type'] = df_assessments_clean['assessment_type'].astype('category')

print(f"\n✓ Assessments cleaned: {len(df_assessments_clean)} records")
print(f"Assessment types: {df_assessments_clean['assessment_type'].nunique()}")



7. CLEANING ASSESSMENTS TABLE
--------------------------------------------------------------------------------
✓ Removed 0 duplicate assessment records

Missing values:
code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

Weight validation:
  Min: 0.0, Max: 100.0
  Sum by module:
code_module
AAA    400.0
BBB    800.0
CCC    600.0
DDD    800.0
EEE    600.0
FFF    800.0
GGG    300.0
Name: weight, dtype: float64

✓ Assessments cleaned: 206 records
Assessment types: 3


In [9]:
# 8. CLEAN STUDENT ASSESSMENT TABLE
print("\n\n8. CLEANING STUDENT ASSESSMENT TABLE")
print("-" * 80)

df_student_assessment_clean = df_student_assessment.copy()

print(f"Initial shape: {df_student_assessment_clean.shape}")

# Check for duplicates (same student, same assessment, same submission date)
initial_rows = len(df_student_assessment_clean)
df_student_assessment_clean = df_student_assessment_clean.drop_duplicates(
    subset=['id_student', 'id_assessment', 'date_submitted'], keep='first')
print(f"✓ Removed {initial_rows - len(df_student_assessment_clean)} duplicate submissions")

print(f"\nMissing values:")
print(df_student_assessment_clean.isnull().sum())

# Handle missing scores
missing_scores = df_student_assessment_clean['score'].isnull().sum()
print(f"\nMissing score values: {missing_scores}")

# Fill missing scores with 0 (indicates no score/not graded)
df_student_assessment_clean['score'] = df_student_assessment_clean['score'].fillna(0)

# Validate score ranges
print(f"\nScore validation:")
print(f"  Min: {df_student_assessment_clean['score'].min()}, Max: {df_student_assessment_clean['score'].max()}")

# Flag invalid scores (should be 0-100 or 0 for missing)
invalid_scores = df_student_assessment_clean[
    (df_student_assessment_clean['score'] < 0) | 
    (df_student_assessment_clean['score'] > 100)
]
print(f"  Invalid scores (out of 0-100): {len(invalid_scores)}")

if len(invalid_scores) > 0:
    print(f"  Score range: {invalid_scores['score'].min()} - {invalid_scores['score'].max()}")

# Standardize is_banked
df_student_assessment_clean['is_banked'] = df_student_assessment_clean['is_banked'].astype('int8')

print(f"\n✓ Student Assessment cleaned: {len(df_student_assessment_clean)} records")
print(f"Score distribution:")
print(df_student_assessment_clean['score'].describe())



8. CLEANING STUDENT ASSESSMENT TABLE
--------------------------------------------------------------------------------
Initial shape: (173912, 5)
✓ Removed 0 duplicate submissions

Missing values:
id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64

Missing score values: 173

Score validation:
  Min: 0.0, Max: 100.0
  Invalid scores (out of 0-100): 0

✓ Student Assessment cleaned: 173912 records
Score distribution:
count    173912.000000
mean         75.724171
std          18.940093
min           0.000000
25%          65.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: score, dtype: float64


In [10]:
# 9. DATA INTEGRATION - CREATE INTEGRATED DATASETS
print("\n\n9. DATA INTEGRATION - Creating Integrated Datasets")
print("-" * 80)

# Integration 1: Student Profile (Student Info + Registration Summary)
print("\n✓ Creating Student Profile Dataset...")

df_student_profile = df_student_info_clean.copy()

# Add registration info
registration_summary = df_student_registration_clean.groupby('id_student').agg({
    'code_module': 'nunique',
    'code_presentation': 'nunique'
}).rename(columns={
    'code_module': 'num_modules',
    'code_presentation': 'num_presentations'
})

df_student_profile = df_student_profile.merge(registration_summary, 
                                              left_on='id_student', 
                                              right_index=True, 
                                              how='left')

print(f"  Student Profile: {len(df_student_profile)} records")
print(f"  Columns: {df_student_profile.shape[1]}")

# Integration 2: Student Course Registration (Student Info + Registration)
print("\n✓ Creating Student-Course Registration Dataset...")

df_student_course_reg = df_student_registration_clean.merge(
    df_student_info_clean[['id_student', 'gender', 'region', 'highest_education', 'age_band', 'final_result']],
    on='id_student',
    how='left'
)

print(f"  Student-Course Registration: {len(df_student_course_reg)} records")

# Integration 3: Assessment Performance by Student-Course
print("\n✓ Creating Assessment Performance Dataset...")

# Merge assessments with student results
df_assessment_perf = df_student_assessment_clean.merge(
    df_assessments_clean[['id_assessment', 'code_module', 'code_presentation', 'assessment_type', 'weight']],
    on='id_assessment',
    how='left'
).merge(
    df_student_info_clean[['id_student', 'final_result']],
    on='id_student',
    how='left'
)

print(f"  Assessment Performance: {len(df_assessment_perf)} records")

# Integration 4: Student VLE Activity Summary by Course
print("\n✓ Creating Student-VLE Activity Dataset...")

df_student_vle_summary = df_student_vle_clean.groupby(['id_student', 'code_module', 'code_presentation']).agg({
    'sum_click': ['sum', 'mean', 'max'],
    'date': ['min', 'max']
}).reset_index()

df_student_vle_summary.columns = ['id_student', 'code_module', 'code_presentation', 
                                   'total_clicks', 'avg_clicks', 'max_clicks_per_day',
                                   'first_activity_date', 'last_activity_date']

print(f"  Student VLE Summary: {len(df_student_vle_summary)} records")

print(f"\n✓ Data Integration Complete!")
print(f"  - Student Profile: {len(df_student_profile)} students")
print(f"  - Student-Course Registrations: {len(df_student_course_reg)} enrollments")
print(f"  - Assessment Results: {len(df_assessment_perf)} submissions")
print(f"  - VLE Activity: {len(df_student_vle_summary)} student-course VLE summaries")



9. DATA INTEGRATION - Creating Integrated Datasets
--------------------------------------------------------------------------------

✓ Creating Student Profile Dataset...
  Student Profile: 28785 records
  Columns: 14

✓ Creating Student-Course Registration Dataset...
  Student-Course Registration: 32593 records

✓ Creating Assessment Performance Dataset...
  Assessment Performance: 173912 records

✓ Creating Student-VLE Activity Dataset...
  Student VLE Summary: 29228 records

✓ Data Integration Complete!
  - Student Profile: 28785 students
  - Student-Course Registrations: 32593 enrollments
  - Assessment Results: 173912 submissions
  - VLE Activity: 29228 student-course VLE summaries


In [12]:
# 10. DATA VALIDATION & QUALITY CHECKS
print("\n\n10. DATA VALIDATION & QUALITY CHECKS (AFTER CLEANING)")
print("-" * 80)

def validate_data_relationships(df_reg, df_info, df_vle, df_assess_raw, df_assess_student):
    """Validate data relationships and integrity"""
    print("\n✓ Referential Integrity Checks:")
    
    # Check students in registration are in student info
    reg_students = set(df_reg['id_student'].unique())
    info_students = set(df_info['id_student'].unique())
    missing_info = reg_students - info_students
    print(f"  Students in registration missing from info: {len(missing_info)}")
    
    # Check VLE students are in registration
    vle_students = set(df_vle['id_student'].unique())
    missing_reg = vle_students - reg_students
    print(f"  Students in VLE missing from registration: {len(missing_reg)}")
    
    # Check assessment students are in info
    assess_students = set(df_assess_student['id_student'].unique())
    missing_info_assess = assess_students - info_students
    print(f"  Students in assessments missing from info: {len(missing_info_assess)}")
    
    # Check modules consistency
    print(f"\n✓ Module Consistency:")
    reg_modules = set(df_reg['code_module'].unique())
    assess_modules = set(df_assess_raw['code_module'].unique())
    vle_modules = set(df_vle['code_module'].unique())
    
    print(f"  Unique modules in registration: {len(reg_modules)}")
    print(f"  Unique modules in assessments: {len(assess_modules)}")
    print(f"  Unique modules in VLE: {len(vle_modules)}")
    print(f"  Module coverage: {len(reg_modules & assess_modules & vle_modules)}/{len(reg_modules)}")
    
    return missing_info, missing_reg, missing_info_assess

missing_info, missing_reg, missing_info_assess = validate_data_relationships(
    df_student_registration_clean,
    df_student_info_clean,
    df_student_vle_clean,
    df_assessments_clean,
    df_student_assessment_clean
)

# Completeness summary
print("\n✓ Data Completeness Summary (After Cleaning):")
print(f"  Student Info: {len(df_student_info_clean)} records, {df_student_info_clean.isnull().sum().sum()} missing values")
print(f"  Student Registration: {len(df_student_registration_clean)} records, {df_student_registration_clean.isnull().sum().sum()} missing values")
print(f"  Student VLE: {len(df_student_vle_clean)} records, {df_student_vle_clean.isnull().sum().sum()} missing values")
print(f"  Assessments: {len(df_assessments_clean)} records, {df_assessments_clean.isnull().sum().sum()} missing values")
print(f"  Student Assessment: {len(df_student_assessment_clean)} records, {df_student_assessment_clean.isnull().sum().sum()} missing values")
print(f"  VLE: {len(df_vle_clean)} records, {df_vle_clean.isnull().sum().sum()} missing values")



10. DATA VALIDATION & QUALITY CHECKS (AFTER CLEANING)
--------------------------------------------------------------------------------

✓ Referential Integrity Checks:
  Students in registration missing from info: 0
  Students in VLE missing from registration: 0
  Students in assessments missing from info: 0

✓ Module Consistency:
  Unique modules in registration: 7
  Unique modules in assessments: 7
  Unique modules in VLE: 7
  Module coverage: 7/7

✓ Data Completeness Summary (After Cleaning):
  Student Info: 28785 records, 971 missing values
  Student Registration: 32593 records, 22542 missing values
  Student VLE: 10655280 records, 0 missing values
  Assessments: 206 records, 0 missing values
  Student Assessment: 173912 records, 0 missing values
  VLE: 6364 records, 0 missing values


In [13]:
# 11. SAVE CLEANED DATA
print("\n\n11. SAVING CLEANED DATA")
print("-" * 80)

# Save cleaned raw tables
df_student_info_clean.to_csv(cleaned_data_path / 'student_info_clean.csv', index=False)
df_student_registration_clean.to_csv(cleaned_data_path / 'student_registration_clean.csv', index=False)
df_student_vle_clean.to_csv(cleaned_data_path / 'student_vle_clean.csv', index=False)
df_assessments_clean.to_csv(cleaned_data_path / 'assessments_clean.csv', index=False)
df_student_assessment_clean.to_csv(cleaned_data_path / 'student_assessment_clean.csv', index=False)
df_vle_clean.to_csv(cleaned_data_path / 'vle_clean.csv', index=False)
df_courses.to_csv(cleaned_data_path / 'courses.csv', index=False)

print("✓ Saved cleaned raw tables:")
print(f"  - student_info_clean.csv")
print(f"  - student_registration_clean.csv")
print(f"  - student_vle_clean.csv")
print(f"  - assessments_clean.csv")
print(f"  - student_assessment_clean.csv")
print(f"  - vle_clean.csv")
print(f"  - courses.csv")

# Save integrated datasets
df_student_profile.to_csv(cleaned_data_path / 'student_profile.csv', index=False)
df_student_course_reg.to_csv(cleaned_data_path / 'student_course_registration.csv', index=False)
df_assessment_perf.to_csv(cleaned_data_path / 'assessment_performance.csv', index=False)
df_student_vle_summary.to_csv(cleaned_data_path / 'student_vle_summary.csv', index=False)

print("\n✓ Saved integrated datasets:")
print(f"  - student_profile.csv")
print(f"  - student_course_registration.csv")
print(f"  - assessment_performance.csv")
print(f"  - student_vle_summary.csv")

print(f"\n✓ Cleaned data saved to: {cleaned_data_path}")

# Summary statistics
print("\n" + "=" * 80)
print("DATA CLEANING & INTEGRATION - FINAL SUMMARY")
print("=" * 80)
print(f"Total students: {len(df_student_info_clean)}")
print(f"Total modules: {df_student_info_clean['id_student'].nunique()}")
print(f"Total registrations: {len(df_student_registration_clean)}")
print(f"Total VLE interactions: {len(df_student_vle_clean)}")
print(f"Total assessment submissions: {len(df_student_assessment_clean)}")
print(f"\nTarget variable distribution:")
print(df_student_info_clean['final_result'].value_counts())
print("\n✓ Data ready for EDA and Feature Engineering!")



11. SAVING CLEANED DATA
--------------------------------------------------------------------------------
✓ Saved cleaned raw tables:
  - student_info_clean.csv
  - student_registration_clean.csv
  - student_vle_clean.csv
  - assessments_clean.csv
  - student_assessment_clean.csv
  - vle_clean.csv
  - courses.csv

✓ Saved integrated datasets:
  - student_profile.csv
  - student_course_registration.csv
  - assessment_performance.csv
  - student_vle_summary.csv

✓ Cleaned data saved to: ..\data\cleaned

DATA CLEANING & INTEGRATION - FINAL SUMMARY
Total students: 28785
Total modules: 28785
Total registrations: 32593
Total VLE interactions: 10655280
Total assessment submissions: 173912

Target variable distribution:
final_result
Pass           10833
Withdrawn       9043
Fail            6264
Distinction     2645
Name: count, dtype: int64

✓ Data ready for EDA and Feature Engineering!
